# Xarray with browser-backed Icechunk I/O

`ipygis` is a bridge to GIS libraries running in the browser. There are a couple of things to know to have it working:

- you must use the `xeus-python` kernel (`ipykernel` currently has a limitation with top-level await and widgets).
- the remote server, or a range-preserving proxy, must allow cross-origin browser requests.
- when using the `@earthmover/icechunk` WASM library, the server must send COOP/COEP headers so that `SharedArrayBuffer` is supported in the browser.

In [1]:
from ipygis.icechunk import Repository, jupyter_storage
from ipygis.zarr import asynchronous as zarr

In [2]:
storage = jupyter_storage("examples/hydrosheds.icechunk")
repository = await Repository.open_async(
    storage,
    # backend="@earthmover/icechunk",  # "icechunk-js" is the default
    proxy_url="https://my-proxy.david-brochart.workers.dev/",
    virtual_chunk_prefixes=["https://data.hydrosheds.org/file/hydrosheds-v2/ACC/1s/"],
)
session = await repository.readonly_session_async("main")
session.snapshot_id

GISWidget()

'A1NX2KVGNEJN6ZX2GR1G'

In [3]:
from ipygis.xarray import open_zarr_async

ds = await open_zarr_async(session.store)
ds
#region = await ds.isel(y=slice(100, 200)).load_async()
#point = await ds.sel(x=10.5, y=48.5, method="nearest").load_async()

<xarray.Dataset> Size: 425GB
Dimensions:  (tile: 41, y: 36000, x: 36000)
Coordinates:
  * tile     (tile) object 328B '0_-100' '20_-120' ... '90_-110' '90_-100'
    tile_x   (tile) float64 328B ...
    tile_y   (tile) float64 328B ...
Dimensions without coordinates: y, x
Data variables:
    0        (tile, y, x) float64 425GB ...

In [4]:
selection = ds["0"].isel(tile=10, y=100, x=200)
result = await selection.load_async()
result

<xarray.DataArray '0' ()> Size: 8B
array(7.)
Coordinates:
    tile     <U7 28B '40_-100'
    tile_x   float64 8B -100.0
    tile_y   float64 8B 40.0
Attributes: (12/30)
    BandName:                    Band_1
    DESCRIPTION:                 Band_1
    DataType:                    Generic
    RepresentationType:          ATHEMATIC
    STATISTICS_COUNT:            7132360.000000
    STATISTICS_COVARIANCES:      8986687.817717677
    ...                          ...
    model_pixel_scale:           [0.0002777777777777778, 0.000277777777777777...
    model_tiepoint:              [0, 0, 0, -100, 0, 0]
    model_type:                  2
    photometric_interpretation:  1
    proj_citation:               ESRI PE String = GEOGCS["GCS_WGS_1984",DATUM...
    raster_type:                 1

In [14]:
ds.tile_y

<xarray.DataArray 'tile_y' (tile: 41)> Size: 328B
array([ 0., 20., 20., 20., 30., 30., 30., 40., 40., 40., 40., 50., 50.,
       50., 50., 60., 60., 60., 60., 60., 60., 60., 60., 70., 70., 70.,
       70., 70., 70., 70., 70., 80., 80., 80., 80., 80., 80., 80., 80.,
       90., 90.])
Coordinates:
  * tile     (tile) object 328B '0_-100' '20_-120' ... '90_-110' '90_-100'
    tile_x   (tile) float64 328B -100.0 -120.0 -110.0 ... -100.0 -110.0 -100.0
    tile_y   (tile) float64 328B 0.0 20.0 20.0 20.0 30.0 ... 80.0 80.0 90.0 90.0

In [6]:
#region = await ds.isel(y=slice(100, 200)).load_async()
#region
point = await ds.sel(x=10.5, y=48.5, method="nearest").load_async()
point

<class 'ValueError'>: cannot supply selection options {'method': 'nearest', 'tolerance': None} for dimension 'x'that has no associated coordinate or index

In [3]:
group = await zarr.open_group(session.store, mode="r")
array = await group.getitem("0")
array.shape, array.dtype, array.chunks

((41, 36000, 36000), dtype('float64'), (1, 512, 512))

In [20]:
result = await array.getitem((10, 100, 200))
result

np.float64(7.0)

In [ ]:
result = await array.getitem((10, 100, 200))
result

In [ ]:
# Close after finishing all array reads.
await session.aclose()
await repository.aclose()